# The low-precision KV cache: FP8 and NVFP4 in vLLM

Companion notebook for the GDE #6 post. All arithmetic is computed here, not pasted from slides — run every cell.

In [ ]:
def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_elem):
    """KV bytes for ONE token: 2 (K and V) x layers x KV heads x head dim x bytes/elem."""
    return 2 * layers * kv_heads * head_dim * bytes_per_elem

MODELS = {
    "Llama-3.1-8B": dict(layers=32, kv_heads=8, head_dim=128),
    "Llama-3.1-70B": dict(layers=80, kv_heads=8, head_dim=128),
}
# NVFP4: 0.5 B/elem for the E2M1 values + 1 B FP8 scale per 16-element block = 0.5625 B/elem
DTYPES = {"BF16": 2.0, "FP8": 1.0, "NVFP4": 0.5625}

for name, m in MODELS.items():
    for dt, bpe in DTYPES.items():
        kib = kv_bytes_per_token(**m, bytes_per_elem=bpe) / 1024
        gib128k = kv_bytes_per_token(**m, bytes_per_elem=bpe) * 131072 / 2**30
        print(f"{name:15s} {dt:6s} {kib:6.1f} KiB/token   128K ctx = {gib128k:5.2f} GiB")

assert abs(kv_bytes_per_token(32, 8, 128, 2.0) / 1024 - 128) < 1e-9
assert abs(kv_bytes_per_token(80, 8, 128, 2.0) * 131072 / 2**30 - 40) < 0.05
assert abs(kv_bytes_per_token(32, 8, 128, 0.5625) / 1024 - 36) < 1e-9
print("OK: 8B BF16 = 128 KiB/token, 70B BF16 @128K = ~40 GiB, 8B NVFP4 = 36 KiB/token")

In [ ]:
# Tokens that fit in a fixed KV pool, and what that means for concurrency.
POOL_GIB = 60.0  # H100 80 GB, 8B BF16 weights (~16 GiB) + overhead leaves ~60 GiB for KV

def max_users(ctx, bpe, pool_gib=POOL_GIB):
    per_req = kv_bytes_per_token(32, 8, 128, bpe) * ctx / 2**30
    return int(pool_gib // per_req)

for ctx in (32768, 131072):
    row = {d: max_users(ctx, b) for d, b in DTYPES.items()}
    print(f"{ctx//1024}K ctx, 8B on one H100: {row}  (NVFP4/BF16 = {row['NVFP4']/row['BF16']:.1f}x)")

assert max_users(131072, 2.0) == 3 and max_users(131072, 1.0) == 7 and max_users(131072, 0.5625) == 13
assert max_users(32768, 2.0) == 15 and max_users(32768, 0.5625) == 53
print("OK: at 128K ctx, FP8 7 users vs BF16 3; NVFP4 13 — 4.3x the concurrency of BF16")

In [ ]:
# The decode break-even: FP8 halves memory AND cuts the decode ITL slope to ~54% of BF16
# (vLLM blog, Apr 2026, Llama-3.1-8B on H100), with break-even around ~7k tokens.
# Toy model: per-token decode time = fixed overhead + slope * bytes read per step.
def decode_itl(ctx, dtype_slope, fixed_ms=1.0, base_slope_ms_per_gib=0.35):
    gib = kv_bytes_per_token(32, 8, 128, DTYPES[dtype_slope] if dtype_slope in DTYPES else 2.0) * ctx / 2**30
    return fixed_ms + base_slope_ms_per_gib * gib

for ctx in (1024, 4096, 8192, 32768, 131072):
    bf16, fp8 = decode_itl(ctx, "BF16"), decode_itl(ctx, "FP8")
    print(f"ctx {ctx:6d}: BF16 {bf16:6.2f} ms/token  FP8 {fp8:6.2f} ms/token  FP8/BF16 = {fp8/bf16:.2f}")
print("Toy model only — the 0.54x slope and ~7k-token break-even are the vLLM blog's measured figures.")

In [ ]:
# Config guard: pick --kv-cache-dtype from the hardware you actually have.
# vLLM 0.32 widened NVFP4 KV to SM8x (Ampere) and SM90 (Hopper); before 0.32 it was SM100-only.
# PR #46963 refuses two combos at backend selection: head_dim 64 (10-15% misreads) and attention sinks.
def kv_cache_dtype(sm, head_dim, attention_sink=False, vllm_version=(0, 32)):
    if attention_sink:
        return None, "refused: attention sinks are incompatible with NVFP4 KV cache (PR #46963)"
    if head_dim == 64:
        return None, "refused: head_dim 64 misreads 10-15% with NVFP4 KV cache (PR #46963)"
    if sm >= 100 or (vllm_version >= (0, 32) and sm in (80, 86, 90)):
        return "nvfp4", "NVFP4 KV cache available"
    if sm in (80, 86, 90):
        return "fp8", "vLLM < 0.32: NVFP4 KV is SM100-only, fall back to FP8"
    return "fp8", "older GPU: FP8 KV cache (software path on A100, no FP8 tensor cores)"

for sm, hd in [(100, 128), (90, 128), (80, 128), (90, 64), (80, 128)]:
    print(f"SM{sm} head_dim={hd}: {kv_cache_dtype(sm, hd)}")
assert kv_cache_dtype(100, 128)[0] == "nvfp4"
assert kv_cache_dtype(90, 128)[0] == "nvfp4"
assert kv_cache_dtype(90, 64)[0] is None
assert kv_cache_dtype(80, 128, vllm_version=(0, 31))[0] == "fp8"
print("OK: config guard matches vLLM 0.32 support matrix")

## Takeaways

1. KV bytes/token = 2 × layers × KV heads × head dim × bytes/elem. FP8 halves it; NVFP4 cuts it to ~28% of BF16.
2. On one H100, an 8B model at 128K context goes from 3 concurrent users (BF16) to 13 (NVFP4) — the flag is a capacity decision, not a micro-optimisation.
3. The accuracy bill is small (<1% for NVFP4 per NVIDIA) but the guard rails are real: no head_dim 64, no attention sinks, and per-head FP8 scales need calibration.
4. Start with `--kv-cache-dtype fp8` on any Hopper/Ampere card; reach for `nvfp4` on vLLM ≥ 0.32 once you have measured your own long-context evals.